# Patient Specific Model

In [1]:
from pathlib import Path
import numpy as np
import h5py

import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, average_precision_score


In [2]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, GridSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score


# 1. Separate Records

> Create a dictionary with the all the data together

In [3]:

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("Features/Seizure").glob("chb01_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

non_files = sorted(
    Path("Features/Non_Seizure").glob("chb01_*_features.mat")
)

for file in non_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]

    if X.shape[0] == 1530:
        X = X.T

    y = np.zeros(X.shape[0], dtype=int)

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(
        "[NON-SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

print("\nTotal records:", len(records))

[SEIZURE] chb01_03 (1437, 1530) (1437,) positives = 16
[SEIZURE] chb01_04 (1437, 1530) (1437,) positives = 11
[SEIZURE] chb01_15 (1437, 1530) (1437,) positives = 16
[SEIZURE] chb01_16 (1437, 1530) (1437,) positives = 21
[SEIZURE] chb01_18 (1437, 1530) (1437,) positives = 37
[SEIZURE] chb01_21 (1437, 1530) (1437,) positives = 38
[SEIZURE] chb01_26 (927, 1530) (927,) positives = 41
[NON-SEIZURE] chb01_01 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_02 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_05 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_06 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_07 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_08 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_09 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_10 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_11 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_12 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_13 (1437, 1530) (

>records = { <br>
    "chb01_03": (X_03, y_03),<br>
    "chb01_04": (X_04, y_04),<br>
    "chb01_05": (X_05, y_05),<br>
    ...<br>
}

## 1.1  Separate Seizure and Non-seizure

> Separatre the dictionary in 2

In [4]:
# =====================================================
# SPLIT SEIZURE / NON-SEIZURE RECORDS
# =====================================================
seizure_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) > 0
}

non_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) == 0
}


print("Seizure:", len(seizure_records))
print("Non seizure:", len(non_records))

Seizure: 7
Non seizure: 30


## 1.2 Pick $R$ Non-seizure Records

In [5]:
# =====================================================
# SELECT NON-SEIZURE RECORDS FOR TRAINING
# =====================================================

r = 0
random.seed(42)


selected_non = random.sample(
    list(non_records.keys()),
    r
)


print("Selected non seizure:")
print(selected_non)

Selected non seizure:
[]


# 2. Training Function

> This will remove the test record from train

In [36]:
def create_train_test(test_record):

    X_train = []
    y_train = []
    train_groups = []


    # -------------------------
    # seizure records
    # -------------------------

    for name, (X,y) in seizure_records.items():

        if name == test_record:
            continue


        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # -------------------------
    # non seizure records
    # -------------------------

    for name in selected_non:

        X,y = non_records[name]

        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # combine

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    train_groups = np.array(train_groups)



    # test

    X_test, y_test = seizure_records[test_record]


    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups
    )

# 3. Model

> Things that we want to tune:</br>
> - k number of features </br>
> - Value of C

In [ ]:
from joblib import Memory


In [46]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, RandomizedSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score

from joblib import Memory
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# =====================================================
# CACHE
# =====================================================

memory = Memory(
    location="./cache",
    verbose=0
)



# =====================================================
# PIPELINE FOR TUNING
# =====================================================

pipeline = Pipeline([

    ("scale", StandardScaler()),

    ("select", SelectPercentile(
        f_classif
    )),

    ("svm", SVC(
        kernel="rbf",
        probability=False
    ))

],
memory=memory)



# =====================================================
# PARAMETERS
# =====================================================

param_grid = {


    "select__percentile":[
        2,
        5,
        10
    ],


    "svm__C":[
        0.1,
        1,
        10
    ],


    "svm__gamma":[
        "scale"
    ]

}



# =====================================================
# OUTER LORO
# =====================================================

results = {}



for test_record in seizure_records.keys():


    print("\n================")
    print("TEST:", test_record)
    print("================")



    (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups

    ) = create_train_test(test_record)



    print(
        "Train:",
        X_train.shape,
        "positives:",
        np.sum(y_train)
    )


    print(
        "Test:",
        X_test.shape,
        "positives:",
        np.sum(y_test)
    )



    # =================================================
    # INNER GROUP CV
    # =================================================

    inner_cv = GroupKFold(
        n_splits=3
    )



    search = RandomizedSearchCV(

        estimator=pipeline,

        param_distributions=param_grid,

        n_iter=9,

        scoring="average_precision",

        cv=inner_cv,

        n_jobs=-1,

        random_state=42

    )



    search.fit(

        X_train,

        y_train,

        groups=train_groups

    )



    print(
        "Best params:",
        search.best_params_
    )



    # =================================================
    # RETRAIN BEST MODEL WITH PROBABILITY
    # =================================================

    best_percentile = search.best_params_[
        "select__percentile"
    ]

    best_C = search.best_params_[
        "svm__C"
    ]

    best_gamma = search.best_params_[
        "svm__gamma"
    ]



    final_model = Pipeline([


        ("scale", StandardScaler()),


        ("select", SelectPercentile(
            f_classif,
            percentile=best_percentile
        )),


        ("svm", SVC(
            kernel="rbf",
            C=best_C,
            gamma=best_gamma,
            probability=True
        ))

    ])



    final_model.fit(
        X_train,
        y_train
    )



    # =================================================
    # TEST
    # =================================================

    y_prob = final_model.predict_proba(
        X_test
    )[:,1]


    # convert probability to class
    y_pred = (y_prob >= 0.5).astype(int)



    # confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()



    # metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    sensitivity = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    specificity = tn / (tn + fp)


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )



    print("Accuracy:", accuracy)

    print("Sensitivity (Recall):", sensitivity)

    print("Specificity:", specificity)

    print("Precision:", precision)

    print("F1:", f1)



    # keep AUC/AUPRC too

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    auprc = average_precision_score(
        y_test,
        y_prob
    )


    print("AUC:", auc)

    print("AUPRC:", auprc)



print("\nDONE")


TEST: chb01_03
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16


Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 2}
Accuracy: 0.988169798190675
Sensitivity (Recall): 0.25
Specificity: 0.9964813511611541
Precision: 0.4444444444444444
F1: 0.32
AUC: 0.9278676988036593
AUPRC: 0.3949116778930749

TEST: chb01_04
Train: (8112, 1530) positives: 169
Test: (1437, 1530) positives: 11
Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 2}
Accuracy: 0.9895615866388309
Sensitivity (Recall): 0.6363636363636364
Specificity: 0.9922861150070126
Precision: 0.3888888888888889
F1: 0.4827586206896552
AUC: 0.9944536529389265
AUPRC: 0.6866061803808841

TEST: chb01_15
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'svm__gamma': 'scale', 'svm__C': 1, 'select__percentile': 10}
Accuracy: 0.9791231732776617
Sensitivity (Recall): 0.125
Specificity: 0.9887403237156932
Precision: 0.1111111111111111
F1: 0.11764705882352941
AUC: 0.44277797325826884
AUPRC: 0.05045535092735745

TEST: chb01_16
Train: (8112, 

In [47]:
import numpy as np
import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import SelectPercentile, f_classif

from sklearn.svm import SVC

from sklearn.model_selection import GroupKFold, RandomizedSearchCV

from sklearn.metrics import roc_auc_score, average_precision_score

from joblib import Memory
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# =====================================================
# CACHE
# =====================================================

memory = Memory(
    location="./cache",
    verbose=0
)



# =====================================================
# PIPELINE FOR TUNING
# =====================================================

pipeline = Pipeline([

    ("scale", StandardScaler()),

    ("select", SelectPercentile(
        f_classif
    )),

    ("svm", SVC(
        kernel="rbf",
        probability=False
    ))

],
memory=memory)



# =====================================================
# PARAMETERS
# =====================================================

param_grid = {


    "select__percentile":[
        2,
        5,
        10
    ],


    "svm__C":[
        0.1,
        1,
        10
    ],


    "svm__gamma":[
        "scale"
    ]

}



# =====================================================
# OUTER LORO
# =====================================================

results = {}



for test_record in seizure_records.keys():


    print("\n================")
    print("TEST:", test_record)
    print("================")



    (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups

    ) = create_train_test(test_record)



    print(
        "Train:",
        X_train.shape,
        "positives:",
        np.sum(y_train)
    )


    print(
        "Test:",
        X_test.shape,
        "positives:",
        np.sum(y_test)
    )



    # =================================================
    # INNER GROUP CV
    # =================================================

    inner_cv = GroupKFold(
        n_splits=3
    )



    search = GridSearchCV(

    estimator=pipeline,

    param_grid=param_grid,

    scoring="average_precision",

    cv=inner_cv,

    n_jobs=-8

)



    search.fit(

        X_train,

        y_train,

        groups=train_groups

    )



    print(
        "Best params:",
        search.best_params_
    )



    # =================================================
    # RETRAIN BEST MODEL WITH PROBABILITY
    # =================================================

    best_percentile = search.best_params_[
        "select__percentile"
    ]

    best_C = search.best_params_[
        "svm__C"
    ]

    best_gamma = search.best_params_[
        "svm__gamma"
    ]



    final_model = Pipeline([


        ("scale", StandardScaler()),


        ("select", SelectPercentile(
            f_classif,
            percentile=best_percentile
        )),


        ("svm", SVC(
            kernel="rbf",
            C=best_C,
            gamma=best_gamma,
            probability=True
        ))

    ])



    final_model.fit(
        X_train,
        y_train
    )



    # =================================================
    # TEST
    # =================================================

    y_prob = final_model.predict_proba(
        X_test
    )[:,1]


    # convert probability to class
    y_pred = (y_prob >= 0.5).astype(int)



    # confusion matrix

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()



    # metrics

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    sensitivity = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    specificity = tn / (tn + fp)


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )



    print("Accuracy:", accuracy)

    print("Sensitivity (Recall):", sensitivity)

    print("Specificity:", specificity)

    print("Precision:", precision)

    print("F1:", f1)



    # keep AUC/AUPRC too

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    auprc = average_precision_score(
        y_test,
        y_prob
    )


    print("AUC:", auc)

    print("AUPRC:", auprc)



print("\nDONE")


TEST: chb01_03
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}
Accuracy: 0.988169798190675
Sensitivity (Recall): 0.25
Specificity: 0.9964813511611541
Precision: 0.4444444444444444
F1: 0.32
AUC: 0.9278676988036593
AUPRC: 0.3949116778930749

TEST: chb01_04
Train: (8112, 1530) positives: 169
Test: (1437, 1530) positives: 11
Best params: {'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}
Accuracy: 0.988865692414753
Sensitivity (Recall): 0.6363636363636364
Specificity: 0.9915848527349228
Precision: 0.3684210526315789
F1: 0.4666666666666667
AUC: 0.9944536529389265
AUPRC: 0.6866061803808841

TEST: chb01_15
Train: (8112, 1530) positives: 164
Test: (1437, 1530) positives: 16
Best params: {'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}
Accuracy: 0.9791231732776617
Sensitivity (Recall): 0.125
Specificity: 0.9887403237156932
Precision: 0.1111111111111111
F1: 0.11764705882352941

In [13]:
import numpy as np
import pandas as pd
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0     # seconds
STEP       = 2.5      # seconds  (75% overlap)
GRID_FS    = 1        # scoring grid resolution (Hz). 1 = one label/second.
THRESHOLD  = 0.5      # operating point on y_prob (calibrate later)
K_PERSIST  = 2        # consecutive positive windows required to raise an alarm

SEIZ = pd.read_csv("seizure_annotations.csv")

# =====================================================
# STEP 1: window probabilities -> per-second hypothesis mask
# any-overlap pooling: a second is positive if ANY window covering it,
# that also passes the persistence rule, is positive.
# =====================================================
def windows_to_event_mask(y_prob, duration_sec):
    """y_prob: array, one score per window, in record order.
       returns a per-second boolean mask of length round(duration_sec*GRID_FS)."""
    pos = (y_prob >= THRESHOLD).astype(int)

    # persistence: keep only windows that are part of a run of >= K_PERSIST
    alarm_win = np.zeros_like(pos)
    run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= K_PERSIST:
            alarm_win[i - K_PERSIST + 1 : i + 1] = 1  # backfill the whole run

    n_grid = int(round(duration_sec * GRID_FS))
    mask = np.zeros(n_grid, dtype=bool)
    for i, a in enumerate(alarm_win):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n_grid)] = True
    return mask, n_grid


def ref_mask(record, n_grid):
    """ground-truth per-second mask from seizure_annotations.csv"""
    mask = np.zeros(n_grid, dtype=bool)
    rows = SEIZ[SEIZ["file"] == record]
    for _, r in rows.iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n_grid)] = True
    return mask

# =====================================================
# STEP 2: score every record, accumulate TP/FP/FN
# fold_outputs: list of (record_id, y_prob, duration_sec)
#   include ALL test records (seizure-free too) for honest FP count.
# =====================================================
def score_event_level(fold_outputs):
    TP = FP = FN = 0
    per_record = []
    for record, y_prob, duration_sec in fold_outputs:
        hyp_mask, n_grid = windows_to_event_mask(y_prob, duration_sec)
        ref = ref_mask(record, n_grid)

        ref_ann = Annotation(ref, GRID_FS)
        hyp_ann = Annotation(hyp_mask, GRID_FS)
        s = scoring.EventScoring(ref_ann, hyp_ann)   # SzCORE defaults

        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        per_record.append((record, s.tp, s.fp, s.refTrue - s.tp, duration_sec))

    sens = TP / (TP + FN) if (TP + FN) else 0.0
    prec = TP / (TP + FP) if (TP + FP) else 0.0
    f1   = 2 * sens * prec / (sens + prec) if (sens + prec) else 0.0
    total_days = sum(d for _, _, _, _, d in per_record) / 86400
    fp_day = FP / total_days if total_days else 0.0

    print("===== EVENT-LEVEL (SzCORE) =====")
    print(f"TP={TP}  FP={FP}  FN={FN}")
    print(f"Sensitivity : {sens:.3f}")
    print(f"Precision   : {prec:.3f}")
    print(f"F1          : {f1:.3f}")
    print(f"FP/day      : {fp_day:.3f}")
    return dict(TP=TP, FP=FP, FN=FN, sens=sens, prec=prec, f1=f1, fp_day=fp_day,
                per_record=per_record)

In [14]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1
THRESHOLD  = 0.5      # operating point (calibrate later on OOF train scores)
K_PERSIST  = 2        # consecutive positive windows to raise an alarm

SEIZ = pd.read_csv("seizure_annotations.csv")

param_grid = {
    "select__percentile": [2, 5, 10],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale"],
}

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf", probability=True)),
    ])

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# STEP A: train one model per LORO fold over SEIZURE records.
# Each seizure record is held out once -> gives its test scores.
# Also fit ONE model on ALL seizure records to score the
# seizure-free records (never in training -> no leakage).
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_

fold_outputs = []     # (record, y_prob, y_true, duration) for ALL records
seg_scores, seg_labels = [], []   # pooled out-of-fold, for segment metrics

# --- seizure records: leave-one-record-out ---
for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best = tune_fit(Xtr, ytr, grps)
    Xte, yte = seizure_records[test_record]
    prob = model.predict_proba(Xte)[:, 1]
    fold_outputs.append((test_record, prob, yte, duration_of(yte)))
    seg_scores.append(prob); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  best={best}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model trained on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _ = tune_fit(Xall, yall, gall)
for name, (X, y) in non_records.items():
    prob = full_model.predict_proba(X)[:, 1]
    fold_outputs.append((name, prob, y, duration_of(y)))
    seg_scores.append(prob); seg_labels.append(y)
    print(f"[NON ] {name}  scored for FP")

# =====================================================
# STEP B: SEGMENT-LEVEL (pooled out-of-fold)
# =====================================================
S = np.concatenate(seg_scores)
L = np.concatenate(seg_labels)
seg_pred = (S >= THRESHOLD).astype(int)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC       : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC     : {roc_auc_score(L, S):.3f}")
print(f"F1@{THRESHOLD}     : {f1_score(L, seg_pred, zero_division=0):.3f}")
print(f"Prevalence  : {L.mean():.4f}")

# =====================================================
# STEP C: EVENT-LEVEL (SzCORE via timescoring)
# =====================================================
def windows_to_mask(y_prob, duration):
    pos = (y_prob >= THRESHOLD).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= K_PERSIST:
            alarm[i - K_PERSIST + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

TP = FP = FN = 0
for record, prob, _, dur in fold_outputs:
    hyp, n = windows_to_mask(prob, dur)
    ref = ref_mask(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp

sens = TP / (TP + FN) if (TP + FN) else 0.0
prec = TP / (TP + FP) if (TP + FP) else 0.0
f1   = 2 * sens * prec / (sens + prec) if (sens + prec) else 0.0
days = sum(d for _, _, _, d in fold_outputs) / 86400
print("\n===== EVENT-LEVEL (SzCORE) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"FP/day      : {FP/days:.3f}")

[SEIZ] chb01_03  best={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=16
[SEIZ] chb01_04  best={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=11
[SEIZ] chb01_15  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=16
[SEIZ] chb01_16  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=21
[SEIZ] chb01_18  best={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=37
[SEIZ] chb01_21  best={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=38
[SEIZ] chb01_26  best={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  test_pos=41
[NON ] chb01_01  scored for FP
[NON ] chb01_02  scored for FP
[NON ] chb01_05  scored for FP
[NON ] chb01_06  scored for FP
[NON ] chb01_07  scored for FP
[NON ] chb01_08  scored for FP
[NON ] chb01_09  scored for FP
[NON ] chb01_10  scored for FP
[NON ] chb01_11  scored for FP
[NON ] chb01_12  scored for FP
[NON ] c

In [16]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# calibration search grids (applied to OOF TRAIN scores only)
THRESH_GRID = None            # None -> derived per fold from score quantiles
K_GRID      = [1, 2, 3, 4, 6, 8]
FP_PER_DAY_BUDGET = 1.9       # Shoeb 0.08 FP/h = 1.92 FP/day ; a-priori ceiling

param_grid = {
    "select__percentile": [1, 2, 5, 10],
    "svm__C": [0.01,0.1, 1, 10],
    "svm__gamma": ["scale",0.01, 0.1],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf")),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    """record_scores: list of (record, scores, y, duration).
       returns total TP, FP, FN over those records."""
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Sweep (threshold, k); pick max sensitivity s.t. FP/day <= budget.
# Tie / infeasible -> fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        qs = np.quantile(all_scores, np.linspace(0.50, 0.999, 25))
        thr_grid = np.unique(qs)
    else:
        thr_grid = THRESH_GRID

    best = None  # (sens, -fp_day, f1, thr, k)
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            cand = (sens, -fp_day, f1, thr, k)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or cand > best_feasible:
                    best_feasible = cand
            if best is None or (f1, sens) > (best[2], best[0]):
                best = cand
    chosen = best_feasible if best_feasible is not None else best
    return chosen[3], chosen[4]   # threshold, k

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(search_estimator, exclude):
    """Recompute OOF scores per TRAINING seizure record using GroupKFold,
    so calibration sees scores the model didn't train on."""
    names = [n for n in seizure_records if n != exclude]
    out = []
    for name in names:
        Xtr, ytr, grps = build_train(exclude=None)  # all seizure records
        # mask: train on all-but-this-name, score this-name
        m_test = (grps == name)
        m_train = ~m_test & (grps != exclude)
        est = make_pipe().set_params(**{k.replace("estimator__",""):v
                                        for k,v in search_estimator.get_params().items()
                                        if k in ("select__percentile","svm__C","svm__gamma")})
        est.fit(Xtr[m_train], ytr[m_train])
        sc = est.decision_function(Xtr[m_test])
        y_name = ytr[m_test]
        out.append((name, sc, y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []        # (record, scores, y, dur) for ALL records (test)
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    # --- calibrate (threshold,k) on OOF TRAIN scores, NOT on test ---
    oof = oof_train_record_scores(search.best_estimator_, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    # --- apply to held-out test record ---
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _, _ = tune_fit(Xall, yall, gall)
# calibrate a single (thr,k) for FP scoring from full OOF
oof_full = oof_train_record_scores(full_model, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF, threshold-free diagnostics)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL (per-record calibrated thr,k)
# =====================================================
TP = FP = FN = 0
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"FP/day      : {FP/days:.3f}")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 2, 'svm__C': 10, 'svm__gamma': 0.01}  thr=-0.306 k=8  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=1.016 k=2  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.965 k=8  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.849 k=8  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.638 k=8  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.492 k=8  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 0.01}  thr=-0.628 k=8  test_pos=41

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.444
ROC-AUC : 0.905
Prev    : 0.0035

===== EVENT-LEVEL (SzCORE, calibrated) =====
TP=6  FP=1  FN=1
Sensitivity : 0.857
Precision   : 0.857
F1          : 0.857
FP/d

In [18]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
          "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"

import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score
from timescoring import scoring
from timescoring.annotations import Annotation

# =====================================================
# CONFIG
# =====================================================
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# calibration search grids (applied to OOF TRAIN scores only)
THRESH_GRID = None            # None -> derived per fold from score quantiles
K_MAX       = 3               # clinical cap: k<=3 -> persistence <= 7.5s (~Shoeb's 6s)
K_GRID      = [k for k in [1, 2, 3, 4, 6, 8] if k <= K_MAX]
FP_PER_DAY_BUDGET = 1.9       # Shoeb 0.08 FP/h = 1.92 FP/day ; a-priori ceiling

param_grid = {
    "select__percentile": [2, 5, 10],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale"],
}

SEIZ = pd.read_csv("seizure_annotations.csv")
_memory = Memory(location="./cache", verbose=0)

def make_pipe():
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif)),
        ("svm",    SVC(kernel="rbf")),
    ], memory=_memory)

def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN

# =====================================================
# EVENT SCORING HELPERS
# =====================================================
def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1 : i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n

def ref_mask_from_record(record, n):
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round(r["start_sec"] * GRID_FS))
        e = int(round(r["end_sec"]   * GRID_FS))
        mask[s:min(e, n)] = True
    return mask

def event_counts(record_scores, threshold, k):
    """record_scores: list of (record, scores, y, duration).
       returns total TP, FP, FN over those records."""
    TP = FP = FN = 0
    for record, scores, y, dur in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN

# =====================================================
# PER-FOLD CALIBRATION on OOF TRAIN scores
# Sweep (threshold, k); pick max sensitivity s.t. FP/day <= budget.
# Tie / infeasible -> fall back to best F1.
# =====================================================
def calibrate(oof_record_scores):
    total_days = sum(d for _, _, _, d in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        qs = np.quantile(all_scores, np.linspace(0.50, 0.999, 25))
        thr_grid = np.unique(qs)
    else:
        thr_grid = THRESH_GRID

    best = None  # (sens, -fp_day, f1, thr, k)
    best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            cand = (sens, -fp_day, f1, thr, k)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or cand > best_feasible:
                    best_feasible = cand
            if best is None or (f1, sens) > (best[2], best[0]):
                best = cand
    chosen = best_feasible if best_feasible is not None else best
    return chosen[3], chosen[4]   # threshold, k

# =====================================================
# BUILD TRAIN over seizure records
# =====================================================
def build_train(exclude=None):
    Xs, ys, grps = [], [], []
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        Xs.append(X); ys.append(y); grps.extend([name] * len(y))
    return np.vstack(Xs), np.concatenate(ys), np.array(grps)

def tune_fit(X, y, groups):
    inner = GroupKFold(n_splits=3)
    search = GridSearchCV(make_pipe(), param_grid,
                          scoring="average_precision", cv=inner, n_jobs=-1)
    search.fit(X, y, groups=groups)
    return search.best_estimator_, search.best_params_, search

def oof_train_record_scores(search_estimator, exclude):
    """Recompute OOF scores per TRAINING seizure record using GroupKFold,
    so calibration sees scores the model didn't train on."""
    names = [n for n in seizure_records if n != exclude]
    out = []
    for name in names:
        Xtr, ytr, grps = build_train(exclude=None)  # all seizure records
        # mask: train on all-but-this-name, score this-name
        m_test = (grps == name)
        m_train = ~m_test & (grps != exclude)
        est = make_pipe().set_params(**{k.replace("estimator__",""):v
                                        for k,v in search_estimator.get_params().items()
                                        if k in ("select__percentile","svm__C","svm__gamma")})
        est.fit(Xtr[m_train], ytr[m_train])
        sc = est.decision_function(Xtr[m_test])
        y_name = ytr[m_test]
        out.append((name, sc, y_name, duration_of(y_name)))
    return out

# =====================================================
# MAIN LORO
# =====================================================
test_outputs = []        # (record, scores, y, dur) for ALL records (test)
seg_scores, seg_labels = [], []
fold_choices = []

for test_record in seizure_records.keys():
    Xtr, ytr, grps = build_train(exclude=test_record)
    model, best, search = tune_fit(Xtr, ytr, grps)

    # --- calibrate (threshold,k) on OOF TRAIN scores, NOT on test ---
    oof = oof_train_record_scores(search.best_estimator_, exclude=test_record)
    thr, k = calibrate(oof)
    fold_choices.append((test_record, thr, k))

    # --- apply to held-out test record ---
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    test_outputs.append((test_record, sc, yte, duration_of(yte), thr, k))
    seg_scores.append(sc); seg_labels.append(yte)
    print(f"[SEIZ] {test_record}  params={best}  thr={thr:.3f} k={k}  test_pos={int(yte.sum())}")

# --- seizure-free records: scored by model on ALL seizure records ---
Xall, yall, gall = build_train(exclude=None)
full_model, _, _ = tune_fit(Xall, yall, gall)
# calibrate a single (thr,k) for FP scoring from full OOF
oof_full = oof_train_record_scores(full_model, exclude=None)
thr_fp, k_fp = calibrate(oof_full)
for name, (X, y) in non_records.items():
    sc = full_model.decision_function(X)
    test_outputs.append((name, sc, y, duration_of(y), thr_fp, k_fp))
    seg_scores.append(sc); seg_labels.append(y)

# =====================================================
# SEGMENT-LEVEL (pooled OOF, threshold-free diagnostics)
# =====================================================
S = np.concatenate(seg_scores); L = np.concatenate(seg_labels)
print("\n===== SEGMENT-LEVEL (pooled OOF) =====")
print(f"AUPRC   : {average_precision_score(L, S):.3f}")
print(f"ROC-AUC : {roc_auc_score(L, S):.3f}")
print(f"Prev    : {L.mean():.4f}")

# =====================================================
# EVENT-LEVEL (per-record calibrated thr,k)
# =====================================================
# =====================================================
# LATENCY: onset -> end of alarm-defining window (k-th consecutive positive)
# Measured only for DETECTED seizures (latency is undefined for misses).
# =====================================================
def detection_latencies(record, sc, dur, thr, k):
    """For each seizure in `record`, find the first alarm-defining window whose
    end falls at/after onset and within (onset - tol_pre, offset + tol_post).
    Returns list of latencies in seconds (one per detected seizure)."""
    TOL_PRE, TOL_POST = 30, 60        # SzCORE matching tolerances
    pos = (sc >= thr).astype(int)
    # index of the k-th window that completes each run (the alarm-defining window)
    alarm_end_times = []
    run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:                              # alarm raised exactly here
            alarm_end_times.append((i * STEP) + WINDOW_LEN)   # end-time of this window
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in alarm_end_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)        # earliest matching alarm
    return lat

TP = FP = FN = 0
all_latencies = []
for record, sc, y, dur, thr, k in test_outputs:
    hyp, n = windows_to_mask(sc, dur, thr, k)
    ref = ref_mask_from_record(record, n)
    s = scoring.EventScoring(Annotation(ref, GRID_FS), Annotation(hyp, GRID_FS))
    TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    all_latencies.extend(detection_latencies(record, sc, dur, thr, k))

sens = TP/(TP+FN) if (TP+FN) else 0.0
prec = TP/(TP+FP) if (TP+FP) else 0.0
f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
days = sum(t[3] for t in test_outputs) / 86400
med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")
print("\n===== EVENT-LEVEL (SzCORE, calibrated) =====")
print(f"TP={TP}  FP={FP}  FN={FN}")
print(f"Sensitivity   : {sens:.3f}")
print(f"Precision     : {prec:.3f}")
print(f"F1            : {f1:.3f}")
print(f"FP/day        : {FP/days:.3f}")
print(f"Median latency: {med_lat:.1f}s  (n={len(all_latencies)} detected)")
print("\nPer-fold (threshold, k):")
for r, t, k in fold_choices:
    print(f"  {r}: thr={t:.3f}  k={k}")

[SEIZ] chb01_03  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.712 k=1  test_pos=16
[SEIZ] chb01_04  params={'select__percentile': 2, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.726 k=1  test_pos=11
[SEIZ] chb01_15  params={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=1.090 k=1  test_pos=16
[SEIZ] chb01_16  params={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.979 k=3  test_pos=21
[SEIZ] chb01_18  params={'select__percentile': 10, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.642 k=1  test_pos=37
[SEIZ] chb01_21  params={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.725 k=1  test_pos=38
[SEIZ] chb01_26  params={'select__percentile': 5, 'svm__C': 1, 'svm__gamma': 'scale'}  thr=0.663 k=1  test_pos=41

===== SEGMENT-LEVEL (pooled OOF) =====
AUPRC   : 0.386
ROC-AUC : 0.912
Prev    : 0.0035

===== EVENT-LEVEL (SzCORE, calibrated) =====
TP=4  FP=0  FN=3
Sensitivity   : 0.571
Precision     : 1.000
F1 